# TCV ensemble + calibrated uncertainty

Per-timestep confinement state classification, to low-confinement (L), dithering (D) or high-confinement mode (H).

We will:
- train XGBoost and a 1D CNN
- build an ensemble by averaging their per-class probabilities
- fit temperature scaling for each source
- report ECE / reliability diagrams and a coverage-accuracy curve on the test set

In [ ]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import xgboost as xgb

np.random.seed(0)
torch.manual_seed(0)

ROOT_DIR = Path.cwd()
DATA_PATH_FLAT = str(ROOT_DIR) + "/parquets/"
FILENAME = "TCV_confstate_*.parquet"

in_features = ["Halpha13", "FIR_LIDs_core", "INPWR", "BETAT"]
out_feature = "label_conf"
N_CLASSES = 3
CLASS_NAMES = {0: "L", 1: "D", 2: "H"}

## Shot splits

See `data/download_dataset.ipynb` to download the shots

In [ ]:
shots = glob.glob(DATA_PATH_FLAT + FILENAME)
shots = {int(f.split("TCV_confstate_")[1].split(".parquet")[0]): f for f in shots}

shots_train = [30225, 31839, 57077, 57093, 57732, 58460, 59076, 60097, 60813, 60995, 61000, 61021, 61039, 61246, 61260, 61279, 63308, 64370, 64820, 65564, 66494, 68643, 69514, 69515, 73330, 73631, 73931, 76702, 77602, 79826]
shots_test_smallelm = [78069, 64372]
shots_test_other = [29511, 30262, 43454, 48580, 57013, 59065, 59825, 63246, 63923, 63944, 64386, 65481, 79827]

print(f"train/calibration        : {len(shots_train)}")
print(f"test small-ELM           : {len(shots_test_smallelm)}")
print(f"test other               : {len(shots_test_other)}")

## Load

In [ ]:
def load_shot(shot_id):
    df = pd.read_parquet(shots[shot_id])
    df = df.dropna(subset=in_features + [out_feature, "time"])
    X = df[in_features].to_numpy(dtype=np.float32)
    y = df[out_feature].to_numpy(dtype=np.int64)
    t = df["time"].to_numpy(dtype=np.float32)
    return X, y, t

def stack_shots(shot_list):
    Xs, ys = [], []
    for s in shot_list:
        X, y, _ = load_shot(s)
        Xs.append(X); ys.append(y)
    return np.concatenate(Xs), np.concatenate(ys)

X_train_flat, y_train_flat = stack_shots(shots_train)
feat_mean = X_train_flat.mean(0)
feat_std  = X_train_flat.std(0) + 1e-6

print(f"train rows: {X_train_flat.shape[0]}")
fr = np.bincount(y_train_flat, minlength=N_CLASSES) / len(y_train_flat)
print(f"class fractions (train): " + "  ".join(f"{CLASS_NAMES[c]}={fr[c]:.3f}" for c in range(N_CLASSES)))

# (A) Model training

## XGBoost (per-timeslice)

In [ ]:
xgb_clf = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    objective="multi:softprob", num_class=N_CLASSES,
    n_jobs=-1, tree_method="hist",
)
xgb_clf.fit(X_train_flat, y_train_flat)
print("train acc:", (xgb_clf.predict(X_train_flat) == y_train_flat).mean().round(3))

## 1D CNN (windowed)

In [ ]:
WINDOW = 200  # 20ms

class WindowDataset(Dataset):
    def __init__(self, shot_ids, mean, std, window=WINDOW, stride=1):
        self.window = window
        self.X_list, self.y_list, self.index = [], [], []
        for k, s in enumerate(shot_ids):
            X, y, _ = load_shot(s)
            X = (X - mean) / std
            self.X_list.append(X.astype(np.float32))
            self.y_list.append(y)
            for i in range(window - 1, len(X), stride):
                self.index.append((k, i))

    def __len__(self): return len(self.index)

    def __getitem__(self, idx):
        k, i = self.index[idx]
        x = self.X_list[k][i - self.window + 1 : i + 1]
        return torch.from_numpy(x.T), int(self.y_list[k][i])

class ConfCNN(nn.Module):
    def __init__(self, in_channels, n_classes=N_CLASSES, hidden=32):
        super().__init__()
        def block(c_in, c_out):
            return nn.Sequential(
                nn.Conv1d(c_in, c_out, kernel_size=5, padding=2),
                nn.BatchNorm1d(c_out),
                nn.GELU(),
            )
        self.body = nn.Sequential(
            block(in_channels, hidden),
            block(hidden, hidden),
            block(hidden, hidden),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )
        self.head = nn.Linear(hidden, n_classes)

    def forward(self, x):
        return self.head(self.body(x))

In [ ]:
train_ds = WindowDataset(shots_train, feat_mean, feat_std, stride=1)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0, drop_last=True)
print(f"training windows: {len(train_ds)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn = ConfCNN(in_channels=len(in_features)).to(device)
opt = torch.optim.AdamW(cnn.parameters(), lr=1e-3, weight_decay=1e-4)
ce = nn.CrossEntropyLoss()

EPOCHS = 10
for epoch in range(EPOCHS):
    cnn.train()
    n, correct, loss_sum = 0, 0, 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = cnn(xb)
        loss = ce(logits, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        n += xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        loss_sum += loss.item() * xb.size(0)
    print(f"epoch {epoch}: loss={loss_sum/n:.4f} acc={correct/n:.4f}")

# (B) Evaluation

## Per-shot evaluation

In [ ]:
STRIDE = 10

@torch.no_grad()
def eval_shot(shot_id):
    X, y, t = load_shot(shot_id)
    Xn = ((X - feat_mean) / feat_std).astype(np.float32)
    idx = np.arange(WINDOW - 1, len(X), STRIDE)
    # XGB on the aligned timesteps
    p_xgb = xgb_clf.predict_proba(X[idx])
    # CNN windows ending at idx
    win = np.stack([Xn[i - WINDOW + 1 : i + 1] for i in idx]).transpose(0, 2, 1)
    cnn.eval()
    win = torch.from_numpy(win).to(device)
    p_cnn = []
    for s in range(0, len(win), 512):
        p_cnn.append(torch.softmax(cnn(win[s:s + 512]), 1).cpu().numpy())
    p_cnn = np.concatenate(p_cnn)
    p_ens = (p_xgb + p_cnn) / 2.0
    return {"time": t[idx], "true": y[idx], "P_xgb": p_xgb, "P_cnn": p_cnn, "P_ens": p_ens}

def gather(shot_list):
    Ys, P_x, P_c, P_e = [], [], [], []
    for s in shot_list:
        r = eval_shot(s)
        Ys.append(r["true"]); P_x.append(r["P_xgb"]); P_c.append(r["P_cnn"]); P_e.append(r["P_ens"])
    return (np.concatenate(Ys),
            np.concatenate(P_x), np.concatenate(P_c), np.concatenate(P_e))

Y_cal, P_xgb_cal, P_cnn_cal, P_ens_cal   = gather(shots_train)
Y_test_se,  P_xgb_test_se,  P_cnn_test_se,  P_ens_test_se    = gather(shots_test_smallelm)
Y_test_other, P_xgb_test_other, P_cnn_test_other, P_ens_test_other= gather(shots_test_other)

# combined test set across both groups
Y_test    = np.concatenate([Y_test_se,    Y_test_other])
P_xgb_test = np.concatenate([P_xgb_test_se, P_xgb_test_other])
P_cnn_test = np.concatenate([P_cnn_test_se, P_cnn_test_other])
P_ens_test = np.concatenate([P_ens_test_se, P_ens_test_other])

print(f"calibration timesteps: {len(Y_cal):>7d}")
print(f"test timesteps       : {len(Y_test):>7d}  (small-ELM {len(Y_test_se)} + other {len(Y_test_other)})")

### Accuracy

In [ ]:
def acc(P, Y): return (P.argmax(1) == Y).mean()

rows = []
for name, (Y_, Px, Pc, Pe) in [
    ("small-ELM", (Y_test_se,   P_xgb_test_se,   P_cnn_test_se,   P_ens_test_se)),
    ("other",     (Y_test_other, P_xgb_test_other, P_cnn_test_other, P_ens_test_other)),
    ("all test", (Y_test, P_xgb_test, P_cnn_test, P_ens_test)),
]:
    rows.append({"group": name, "n": len(Y_),
                 "XGB":      f"{acc(Px, Y_):.3f}",
                 "CNN":      f"{acc(Pc, Y_):.3f}",
                 "Ensemble": f"{acc(Pe, Y_):.3f}"})
pd.DataFrame(rows)

### Where do the three models disagree?

Three examples, each picking the most extreme few shots:
- **small-ELM, XGB >> CNN**: shots where the per-timeslice XGB beats the windowed CNN by the largest margin.
- **Other, CNN >> XGB**: shots where the windowed CNN beats XGB by the largest margin.
- **All test, Ensemble >> both**: shots where the ensemble beats the better of the two single models by the largest margin.

In [ ]:
def plot_eval(result, s, data_path, pq_name,
              signal_1="IPLA", signal_2="FIR_LIDs_core", signal_3="Halpha13",
              label_column="label_conf", mode="soft", postfix=""):
    if mode not in ("soft", "hard"):
        raise ValueError(f"mode must be 'soft' or 'hard', got {mode!r}")
    pq = pd.read_parquet(os.path.join(data_path, pq_name.replace("*", str(s))))

    cmap_map = {
        0: (0.55, 0.75, 0.85, 1.0),
        1: (0.60, 0.50, 0.70, 1.0),
        2: (0.90, 0.85, 0.50, 1.0),
        3: (0.90, 0.75, 0.60, 1.0),
    }

    fig = plt.figure(figsize=(10, 5))
    gs = fig.add_gridspec(2, 1, height_ratios=[0.9, 0.1], hspace=0.05)
    ax_top = fig.add_subplot(gs[0])
    ax_bot = fig.add_subplot(gs[1], sharex=ax_top)

    # plot signals on the top axis
    axes = [ax_top, ax_top.twinx(), ax_top.twinx()]
    signals = [signal_1, signal_2, signal_3]
    handles_legend, labels_legend = [], []
    for i, (ax, signal) in enumerate(zip(axes, signals)):
        line, = ax.plot(pq.time, pq[signal], color=f"C{i}", linewidth=1.0)
        handles_legend.append(line)
        labels_legend.append(signal)
        if i == 2:
            ax.set_yticks([])
        else:
            ax.tick_params(axis="y", labelcolor=f"C{i}")

    # predictions as colored background
    proba = np.asarray(result["proba"])  # (N, C)
    n_classes = proba.shape[1]
    class_colors = np.array([cmap_map[k] for k in range(n_classes)])  # (C, 4)
    if mode == "soft":
        pred_rgba = proba @ class_colors  # (N, 4)
    else:  # "hard"
        pred_rgba = class_colors[proba.argmax(axis=1)]  # (N, 4)
    pred_rgba[:, 3] = 1.0
    pred_img = np.stack([pred_rgba, pred_rgba], axis=0)  # (2, N, 4)

    t_pred = np.asarray(result["time"])
    y0, y1 = ax_top.get_ylim()
    ax_top.imshow(pred_img, aspect="auto", origin="lower", zorder=-5, interpolation="nearest",
                  extent=[t_pred[0], t_pred[-1], y0, y1])
    ax_top.set_ylim(y0, y1)

    # ground-truth labels in the thin bottom strip
    labels = pq[label_column].values
    time = pq.time.values
    label_img = np.zeros((2, len(labels), 4))
    for i, l in enumerate(labels):
        if np.isfinite(l) and l in cmap_map:
            label_img[:, i, :] = cmap_map[l]
    ax_bot.imshow(label_img, aspect="auto", origin="lower", interpolation="nearest",
                  extent=[time[0], time[-1], 0, 1])
    ax_bot.set_yticks([])
    ax_bot.set_ylabel("labels", rotation=0, ha="right", va="center", labelpad=10)
    ax_bot.set_xlabel("time")
    plt.setp(ax_top.get_xticklabels(), visible=False)

    # legends on the rightmost twin axis
    class_label_names = {"L": cmap_map[0], "D": cmap_map[1], "H": cmap_map[2]}
    if label_column == "label_conf_qce":
        class_label_names["QCE-H"] = cmap_map[3]
    class_handles = [mpatches.Patch(color=class_label_names[name], label=name)
                     for name in class_label_names]
    legend_1 = axes[2].legend(handles_legend, labels_legend, loc="upper right")
    axes[2].legend(handles=class_handles, loc="upper left")
    axes[2].add_artist(legend_1)

    acc = (np.asarray(result["pred"]) == np.asarray(result["true"])).mean()
    fig.suptitle(f"#{s}, predictions vs labels (acc={acc:.3f}){postfix}", fontsize=16)
    plt.show()

In [ ]:
def _to_eval_dict(r, key):
    P = r[key]
    return {"time": r["time"], "true": r["true"],
            "pred": P.argmax(1), "proba": P}

def per_shot_accs(shot_list):
    out = []
    for s in shot_list:
        r = eval_shot(s)
        out.append({
            "shot": s, "r": r,
            "xgb": (r["P_xgb"].argmax(1) == r["true"]).mean(),
            "cnn": (r["P_cnn"].argmax(1) == r["true"]).mean(),
            "ens": (r["P_ens"].argmax(1) == r["true"]).mean(),
        })
    return out

se_rows    = per_shot_accs(shots_test_smallelm)
other_rows = per_shot_accs(shots_test_other)
all_rows   = se_rows + other_rows

# (header, ranked list, models to plot)
categories = [
    ("small-ELM, XGB > CNN",
     sorted(se_rows,    key=lambda d: d["xgb"] - d["cnn"], reverse=True),
     [("P_xgb", "XGB"), ("P_cnn", "CNN")]),
    ("Other, CNN > XGB",
     sorted(other_rows, key=lambda d: d["cnn"] - d["xgb"], reverse=True),
     [("P_cnn", "CNN"), ("P_xgb", "XGB")]),
    ("all test, Ensemble > both",
     sorted(all_rows,   key=lambda d: d["ens"] - max(d["xgb"], d["cnn"]), reverse=True),
     [("P_xgb", "XGB"), ("P_cnn", "CNN"), ("P_ens", "Ensemble")]),
]

TOP_K = 2

for header, ranked, plots in categories:
    print(f"\n===== {header} =====")
    for pick in ranked[:TOP_K]:
        s = pick["shot"]
        print(f"shot {s}   XGB={pick['xgb']:.3f}   CNN={pick['cnn']:.3f}   ENS={pick['ens']:.3f}")
        for key, label in plots:
            plot_eval(_to_eval_dict(pick["r"], key), s, DATA_PATH_FLAT, FILENAME,
                      mode="hard", postfix=f" [{label}]")

# Calibration

## ECE and reliability

Multi-class analog of the binary calibration story in `synthetic/calibration_ensembling.ipynb`. For a $K$-class problem we use the top-class confidence $\hat p = \max_c P(c)$ and ask: when the model says it is $\hat p$-confident, is it actually right $\hat p$ of the time?

$$ \text{ECE} = \sum_{b=1}^{B} \frac{n_b}{N}\, \left|\, \overline{\hat p}_b - \overline{\mathrm{acc}}_b \,\right| $$

where bin $b$ groups timesteps by top-class confidence, $\overline{\hat p}_b$ is the mean confidence in the bin, and $\overline{\mathrm{acc}}_b$ is the empirical top-1 accuracy. Confidence lives in $[1/K, 1] = [1/3, 1]$.

Temperature scaling: divide the log-probs by a single scalar $T > 0$ and re-softmax:

$$ P_T(c) = \mathrm{softmax}\big(\log P(c) / T\big). $$

We fit $T$ on the calibration shots two ways: minimizing NLL (smooth, standard) and minimizing ECE directly (grid sweep).

In [ ]:
N_BINS = 10
CONF_RANGE = (1 / N_CLASSES, 1.0)  # top-class confidence lives in [1/K, 1]

def bin_stats(confidence, correct, n_bins=N_BINS, conf_range=CONF_RANGE):
    # group predictions into equal-width confidence bins and report,
    # for each bin: mean confidence, empirical top-1 accuracy, count
    lo, hi = conf_range
    bins = np.linspace(lo, hi, n_bins + 1)
    idx = np.clip(np.digitize(confidence, bins) - 1, 0, n_bins - 1)
    avg_p, frac_pos, counts = [], [], []
    for b in range(n_bins):
        m = idx == b
        counts.append(int(m.sum()))
        if m.sum() == 0:
            avg_p.append(np.nan); frac_pos.append(np.nan)
        else:
            avg_p.append(confidence[m].mean())
            frac_pos.append(correct[m].mean())
    return bins, np.array(avg_p), np.array(frac_pos), np.array(counts)

def ece(P, Y, n_bins=N_BINS):
    # weighted average of |mean confidence - empirical top-1 accuracy| per bin
    confidence = P.max(axis=1)
    correct = (P.argmax(axis=1) == Y).astype(float)
    _, avg_p, frac_pos, counts = bin_stats(confidence, correct, n_bins)
    w = counts / counts.sum()
    mask = counts > 0
    return float(np.sum(w[mask] * np.abs(avg_p[mask] - frac_pos[mask])))

def plot_reliability(axes, P, Y, title, color, hist_log=False):
    # top: reliability curve (mean confidence vs empirical top-1 accuracy, diagonal = perfect)
    # bottom: histogram of confidences (where the model spends its mass)
    ax_rel, ax_hist = axes
    confidence = P.max(axis=1)
    correct = (P.argmax(axis=1) == Y).astype(float)
    bins, avg_p, frac_pos, counts = bin_stats(confidence, correct)
    centers = 0.5 * (bins[1:] + bins[:-1])
    lo, hi = CONF_RANGE
    ax_rel.plot([lo, hi], [lo, hi], 'k--', alpha=0.5, label="perfect")
    m = counts > 0
    ax_rel.plot(avg_p[m], frac_pos[m], marker='o', color=color, linewidth=2)
    ax_rel.set_xlim(lo, hi); ax_rel.set_ylim(0, 1)
    ax_rel.set_xlabel("confidence  max P(c)")
    ax_rel.set_ylabel("empirical top-1 accuracy")
    ax_rel.set_title(f"{title}\nECE = {ece(P, Y):.3f}")
    ax_rel.legend(loc='upper left', fontsize=8)
    ax_hist.bar(centers, counts / counts.sum(), width=(hi - lo) / N_BINS,
                color=color, edgecolor='white', alpha=0.85)
    ax_hist.set_xlim(lo, hi)
    ax_hist.set_xlabel("confidence  max P(c)")
    ax_hist.set_ylabel("fraction of timesteps")
    if hist_log:
        ax_hist.set_yscale('log')

def log_probs(P, eps=1e-8):
    return np.log(np.clip(P, eps, 1.0))

def apply_temperature(P, T):
    # softmax(log P / T), implemented stably
    z = log_probs(P) / T
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def fit_temperature_nll(P, Y, n_steps=20, tol=1e-7):
    # standard approach: minimize cross-entropy (= NLL) over T > 0
    # treat log P as the "logits" and divide by T before re-softmax
    # parameterize T = exp(log_T) so the optimizer is unconstrained
    L = torch.tensor(log_probs(P), dtype=torch.float32)
    YY = torch.tensor(Y, dtype=torch.long)
    log_T = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_T], lr=0.1, max_iter=200,
                            tolerance_grad=1e-9, tolerance_change=1e-12)
    ce_loss = nn.CrossEntropyLoss()
    def closure():
        opt.zero_grad()
        loss = ce_loss(L / torch.exp(log_T), YY)
        loss.backward()
        return loss
    prev = float(log_T.item())
    for _ in range(n_steps):
        opt.step(closure)
        cur = float(log_T.item())
        if abs(cur - prev) < tol:
            break
        prev = cur
    return float(torch.exp(log_T).item())

def fit_temperature_ece(P, Y, T_grid=None):
    # alternative: minimize ECE directly on the calibration set.
    # ECE is not smooth in T, so we just sweep a grid.
    if T_grid is None:
        T_grid = np.linspace(0.3, 5.0, 200)
    eces = np.array([ece(apply_temperature(P, T), Y) for T in T_grid])
    return float(T_grid[int(np.argmin(eces))])

### Fit on calibration shots, evaluate on test

In [ ]:
sources = {
    "XGB":      {"P_cal": P_xgb_cal, "P_test": P_xgb_test, "color": "#d62728"},
    "CNN":      {"P_cal": P_cnn_cal, "P_test": P_cnn_test, "color": "#1f77b4"},
    "Ensemble": {"P_cal": P_ens_cal, "P_test": P_ens_test, "color": "#ff7f0e"},
}

params = {}
for name, src in sources.items():
    params[name] = {
        "T_nll": fit_temperature_nll(src["P_cal"], Y_cal),
        "T_ece": fit_temperature_ece(src["P_cal"], Y_cal),
    }
    src["P_test_ts"] = apply_temperature(src["P_test"], params[name]["T_nll"])

print("Fitted temperatures on the calibration set:")
print(f"  {'source':9s}  NLL-fit   ECE-fit")
for name, p in params.items():
    print(f"  {name:9s}   {p['T_nll']:.3f}    {p['T_ece']:.3f}")

print()
print("Test-set ECE:")
for name, src in sources.items():
    print(f"  {name:9s}  raw = {ece(src['P_test'], Y_test):.4f}    "
          f"T-scaled = {ece(src['P_test_ts'], Y_test):.4f}")

### Reliability diagrams

One figure per source. Left column: raw probabilities. Right column: temperature-scaled probabilities. The top row of each figure is the reliability curve (mean confidence vs empirical accuracy, diagonal = perfect); the bottom row is the confidence histogram.

In [ ]:
for name, src in sources.items():
    T = params[name]["T_nll"]
    fig, axes = plt.subplots(2, 2, figsize=(8, 4), constrained_layout=True,
                             gridspec_kw={'height_ratios': [3, 1]})
    plot_reliability((axes[0, 0], axes[1, 0]), src["P_test"],    Y_test,
                     f"{name} (raw)",                 color=src["color"], hist_log=True)
    plot_reliability((axes[0, 1], axes[1, 1]), src["P_test_ts"], Y_test,
                     f"{name} after T-scaling (T={T:.2f})", color=src["color"], hist_log=True)
    plt.show()

## Coverage-accuracy

Rank test timesteps by confidence (most confident first) and plot accuracy on the kept fraction. Shown for each source raw and for the temperature-calibrated ensemble.

In [ ]:
P_ens_test_cal = apply_temperature(P_ens_test, params["Ensemble"]["T_nll"])
P_xgb_test_cal = apply_temperature(P_xgb_test, params["XGB"]["T_nll"])
P_cnn_test_cal = apply_temperature(P_cnn_test, params["CNN"]["T_nll"])

def coverage_curve(P, Y, fractions):
    pred = P.argmax(1); conf = P.max(1)
    order = np.argsort(-conf)
    accs = []
    for f in fractions:
        k = max(1, int(f * len(order)))
        idx = order[:k]
        accs.append((pred[idx] == Y[idx]).mean())
    return accs

fractions = np.linspace(0.05, 1.0, 20)
curves = {
    "XGB (temp)"      : (coverage_curve(P_xgb_test_cal, Y_test, fractions), "#d62728", "--"),
    "CNN (temp)"      : (coverage_curve(P_cnn_test_cal, Y_test, fractions), "#1f77b4", "--"),
    "Ensemble (temp)" : (coverage_curve(P_ens_test_cal, Y_test, fractions), "#ff7f0e", "-"),
}

fig, ax = plt.subplots(figsize=(6, 2.5), constrained_layout=True)
for name, (accs_, color, ls) in curves.items():
    ax.plot(fractions, accs_, ls, marker="o", color=color, linewidth=1.4,
            markersize=4, label=name)
ax.axhline(1.0, color="black", linestyle=":", linewidth=0.6)
ax.set_xlabel("coverage  (kept = most-confident fraction of test timesteps)")
ax.set_ylabel("accuracy on the kept subset")
ax.set_ylim(0.8, 1.02); ax.legend(fontsize=8, loc="lower left")
ax.set_title("Coverage-accuracy on the combined test set")
fig.set_dpi(200)
plt.show()

- The two inductive biases (axis-aligned splits in XGB, conv kernels over time in CNN) miss different timesteps; averaging their probabilities recovers most of the per-group accuracy lost by either alone.
- Raw confidences tend to be overconfident; temperature scaling brings the high-confidence bins closer to the diagonal.
- The calibrated ensemble gives the most useful coverage curve: confidence ordering tracks accuracy, so thresholding it yields a meaningful "I'm not sure" subset.